# Stage 10 — Dataset Preparation for LSTM Flood Risk Classification

## 1. Pendahuluan

Notebook ini mengimplementasikan **Stage 10** dari pipeline penelitian klasifikasi risiko banjir
berbasis LSTM untuk Kota Padang, Sumatera Barat.

**Tujuan Stage 10:**

1. Memvalidasi dataset hasil Stage 9 (`08_labeled_dataset.csv`).
2. Melakukan pembagian dataset secara **kronologis** (bukan random/shuffle/stratified) ke dalam
   subset Train, Validation, dan Test.
3. Melakukan **label encoding** pada kolom `risk_label`.
4. Melakukan **feature scaling** menggunakan `MinMaxScaler` yang di-*fit* hanya pada data Train.
5. Membentuk **sequence** time-series untuk tiga skenario lookback window: **LB7**, **LB14**, dan **LB30**.
6. Melakukan audit menyeluruh terhadap sequence yang terbentuk maupun yang dibuang akibat missing value.
7. Menyimpan seluruh artefak (scaler, array `.npy`, file `.csv`) ke dalam folder `stage10/`.
8. Menghasilkan laporan otomatis `STAGE10_REPORT.md`.

**Sumber kebenaran (input):**

- `08_labeled_dataset.csv` — dataset hasil Stage 8 (pelabelan), yang telah diaudit pada Stage 9.
- `STAGE9_REPORT.md` — laporan audit label dan kesiapan modeling dari tahap sebelumnya.

**Feature model (8 fitur, sesuai hasil audit Stage 9 — kolom `pw` tidak tersedia di dataset dan
`cape` bukan feature model):**

`rr`, `tavg`, `rh`, `cin`, `kindex`, `li`, `tt`, `sweat`

**Target:** `risk_label`

**Batasan metodologi (harus dipatuhi secara harfiah, tidak boleh ditambah/diganti):**

- Split dataset **wajib kronologis**: Train (2017-01-01 s.d. 2022-12-31), Validation (2023-01-01
  s.d. 2023-12-31), Test (2024-01-01 s.d. 2024-12-31). Dilarang random split, shuffle, atau
  stratified split.
- Scaler (`MinMaxScaler`) hanya boleh di-*fit* pada data Train, lalu digunakan untuk men-*transform*
  Train, Validation, dan Test.
- Dataset sumber **tidak boleh diubah** — tidak ada `fillna`, interpolasi, ataupun imputasi apa pun.
  Sequence yang mengandung nilai NaN pada fitur input **dibuang**, dicatat jumlah dan persentasenya.
- Stage ini **tidak** melakukan training model, hyperparameter tuning, class balancing/SMOTE,
  feature selection, maupun evaluasi model (confusion matrix, precision, recall, F1, ROC/AUC, dst).


## Import Library

In [1]:
# Import seluruh library yang dibutuhkan untuk Stage 10
import os
import json
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import joblib

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

print("Library berhasil diimpor.")


Library berhasil diimpor.


## 2. Load Dataset

Memuat dataset hasil Stage 9 (`08_labeled_dataset.csv`) sebagai sumber kebenaran utama.

In [2]:
INPUT_FILE = "08_labeled_dataset.csv"

df = pd.read_csv(INPUT_FILE, parse_dates=["date"])

print(f"Dataset berhasil dimuat: {df.shape[0]} baris, {df.shape[1]} kolom")
df.head()


Dataset berhasil dimuat: 2922 baris, 15 kolom


,date,selected_hour,selection_status,rr,tavg,rh,cin,kindex,li,tt,sweat,cape,missing_group,risk_label,label_source
0,2017-01-01,12Z,SELECTED,26.9,28.2,83.9,-4.906,34.4,-4.510,42.2,219.162,2226.401,IMPUTED_OGIMET,Sedang,RR_ONLY
1,2017-01-02,12Z,SELECTED,26.9,26.5,87.4,-22.887,35.2,-2.955,41.5,223.381,921.164,ORIGINAL,Sedang,RR_ONLY
2,2017-01-03,12Z,SELECTED,78.0,26.2,87.6,0.000,36.4,-3.317,40.5,277.368,1671.572,ORIGINAL,Tinggi,RR_ONLY
3,2017-01-04,12Z,SELECTED,33.0,24.7,92.4,-4.865,35.1,-0.752,39.0,294.560,397.409,ORIGINAL,Sedang,RR_ONLY
4,2017-01-05,12Z,SELECTED,83.0,25.7,88.3,-15.811,38.0,-5.124,44.1,299.131,2064.721,ORIGINAL,Tinggi,RR_ONLY


In [3]:
# Kolom feature model (8 kolom, sesuai spesifikasi — cape TIDAK digunakan sebagai feature)
FEATURE_COLUMNS = ["rr", "tavg", "rh", "cin", "kindex", "li", "tt", "sweat"]
TARGET_COLUMN = "risk_label"

print("Jumlah feature model:", len(FEATURE_COLUMNS))
print("Feature model:", FEATURE_COLUMNS)
print("Target:", TARGET_COLUMN)


Jumlah feature model: 8
Feature model: ['rr', 'tavg', 'rh', 'cin', 'kindex', 'li', 'tt', 'sweat']
Target: risk_label


## 3. Validasi Dataset

Melakukan validasi terhadap: jumlah baris, jumlah kolom, tipe data, rentang tanggal, duplicate row,
duplicate date, missing value per kolom, dan distribusi label.

In [4]:
validation_results = {}

# Jumlah baris & kolom
validation_results["n_rows"] = int(df.shape[0])
validation_results["n_cols"] = int(df.shape[1])

print(f"Jumlah baris  : {validation_results['n_rows']}")
print(f"Jumlah kolom  : {validation_results['n_cols']}")


Jumlah baris  : 2922
Jumlah kolom  : 15


In [5]:
# Tipe data per kolom
validation_results["dtypes"] = {col: str(dtype) for col, dtype in df.dtypes.items()}

dtypes_df = pd.DataFrame(
    {"kolom": list(validation_results["dtypes"].keys()),
     "tipe_data": list(validation_results["dtypes"].values())}
)
dtypes_df


,kolom,tipe_data
0,date,datetime64[us]
1,selected_hour,str
2,selection_status,str
3,rr,float64
4,tavg,float64
5,rh,float64
6,cin,float64
7,kindex,float64
8,li,float64
9,tt,float64


In [6]:
# Rentang tanggal
validation_results["date_min"] = str(df["date"].min().date())
validation_results["date_max"] = str(df["date"].max().date())

expected_days = (df["date"].max() - df["date"].min()).days + 1
validation_results["expected_days"] = int(expected_days)
validation_results["date_gap"] = int(expected_days - df["date"].nunique())

print(f"Rentang tanggal   : {validation_results['date_min']} s.d. {validation_results['date_max']}")
print(f"Jumlah hari harapan (kalender): {expected_days}")
print(f"Jumlah tanggal unik           : {df['date'].nunique()}")
print(f"Gap tanggal (hari hilang)     : {validation_results['date_gap']}")


Rentang tanggal   : 2017-01-01 s.d. 2024-12-31
Jumlah hari harapan (kalender): 2922
Jumlah tanggal unik           : 2922
Gap tanggal (hari hilang)     : 0


In [7]:
# Duplicate row & duplicate date
validation_results["duplicate_rows"] = int(df.duplicated().sum())
validation_results["duplicate_dates"] = int(df["date"].duplicated().sum())

print(f"Jumlah duplicate row  : {validation_results['duplicate_rows']}")
print(f"Jumlah duplicate date : {validation_results['duplicate_dates']}")


Jumlah duplicate row  : 0
Jumlah duplicate date : 0


In [8]:
# Missing value per kolom
missing_counts = df.isna().sum()
missing_pct = (missing_counts / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    "kolom": missing_counts.index,
    "missing": missing_counts.values,
    "persen": missing_pct.values,
})
validation_results["missing_per_column"] = {
    col: {"missing": int(missing_counts[col]), "persen": float(missing_pct[col])}
    for col in df.columns
}

missing_df


,kolom,missing,persen
0,date,0,0.00
1,selected_hour,0,0.00
2,selection_status,0,0.00
3,rr,0,0.00
4,tavg,0,0.00
5,rh,0,0.00
6,cin,148,5.07
7,kindex,148,5.07
8,li,148,5.07
9,tt,148,5.07


In [9]:
# Distribusi label (risk_label)
label_dist = df["risk_label"].value_counts()
label_dist_pct = (label_dist / len(df) * 100).round(2)

label_dist_df = pd.DataFrame({
    "risk_label": label_dist.index,
    "jumlah": label_dist.values,
    "persen": label_dist_pct.values,
})
validation_results["label_distribution"] = {
    label: {"jumlah": int(label_dist[label]), "persen": float(label_dist_pct[label])}
    for label in label_dist.index
}

label_dist_df


,risk_label,jumlah,persen
0,Rendah,2331,79.77
1,Sedang,372,12.73
2,Tinggi,174,5.95
3,Sangat Tinggi,45,1.54


## 4. Split Kronologis

Membagi dataset ke dalam tiga subset berdasarkan rentang tanggal (bukan random/shuffle/stratified),
sesuai spesifikasi:

- **Train**: 2017-01-01 s.d. 2022-12-31
- **Validation**: 2023-01-01 s.d. 2023-12-31
- **Test**: 2024-01-01 s.d. 2024-12-31

In [10]:
TRAIN_START, TRAIN_END = "2017-01-01", "2022-12-31"
VAL_START, VAL_END = "2023-01-01", "2023-12-31"
TEST_START, TEST_END = "2024-01-01", "2024-12-31"

# Dataset diurutkan berdasarkan tanggal (bukan diacak) untuk menjamin split kronologis
df_sorted = df.sort_values("date").reset_index(drop=True)

train_df = df_sorted[
    (df_sorted["date"] >= TRAIN_START) & (df_sorted["date"] <= TRAIN_END)
].reset_index(drop=True)

validation_df = df_sorted[
    (df_sorted["date"] >= VAL_START) & (df_sorted["date"] <= VAL_END)
].reset_index(drop=True)

test_df = df_sorted[
    (df_sorted["date"] >= TEST_START) & (df_sorted["date"] <= TEST_END)
].reset_index(drop=True)

split_summary = {
    "train": {
        "n_rows": int(len(train_df)),
        "date_min": str(train_df["date"].min().date()),
        "date_max": str(train_df["date"].max().date()),
    },
    "validation": {
        "n_rows": int(len(validation_df)),
        "date_min": str(validation_df["date"].min().date()),
        "date_max": str(validation_df["date"].max().date()),
    },
    "test": {
        "n_rows": int(len(test_df)),
        "date_min": str(test_df["date"].min().date()),
        "date_max": str(test_df["date"].max().date()),
    },
}

for split_name, info in split_summary.items():
    print(f"{split_name:12s}: {info['n_rows']:5d} baris | {info['date_min']} s.d. {info['date_max']}")

# Verifikasi tidak ada tumpang-tindih baris antar subset
assert len(train_df) + len(validation_df) + len(test_df) == len(df_sorted), \
    "Jumlah baris hasil split tidak sama dengan jumlah baris dataset asli!"
print("\nVerifikasi: total baris hasil split sama dengan total baris dataset asli. OK.")


train       :  2191 baris | 2017-01-01 s.d. 2022-12-31
validation  :   365 baris | 2023-01-01 s.d. 2023-12-31
test        :   366 baris | 2024-01-01 s.d. 2024-12-31

Verifikasi: total baris hasil split sama dengan total baris dataset asli. OK.


## 5. Label Encoding

Melakukan encoding pada kolom `risk_label` menggunakan mapping tetap yang telah ditentukan:

| Label | Kode |
|---|---|
| Rendah | 0 |
| Sedang | 1 |
| Tinggi | 2 |
| Sangat Tinggi | 3 |

In [11]:
LABEL_MAPPING = {"Rendah": 0, "Sedang": 1, "Tinggi": 2, "Sangat Tinggi": 3}
INV_LABEL_MAPPING = {v: k for k, v in LABEL_MAPPING.items()}

for subset in (train_df, validation_df, test_df):
    subset["risk_label_encoded"] = subset["risk_label"].map(LABEL_MAPPING)

# Validasi tidak ada label yang gagal ter-encode (NaN)
for name, subset in (("train", train_df), ("validation", validation_df), ("test", test_df)):
    n_unmapped = subset["risk_label_encoded"].isna().sum()
    assert n_unmapped == 0, f"Terdapat {n_unmapped} label yang gagal di-encode pada subset {name}!"

print("Label mapping:", LABEL_MAPPING)
print("Seluruh label pada Train/Validation/Test berhasil di-encode tanpa nilai kosong.")


Label mapping: {'Rendah': 0, 'Sedang': 1, 'Tinggi': 2, 'Sangat Tinggi': 3}
Seluruh label pada Train/Validation/Test berhasil di-encode tanpa nilai kosong.


## 6. Feature Scaling

Menggunakan `MinMaxScaler`. Sesuai aturan, scaler **hanya di-fit pada data Train**, kemudian
digunakan untuk men-*transform* Train, Validation, dan Test. Dilarang melakukan fit pada Validation
atau Test.

Nilai NaN pada fitur sounding (`cin`, `kindex`, `li`, `tt`, `sweat`) diabaikan oleh `MinMaxScaler`
saat proses fit maupun transform, dan tetap dipertahankan sebagai NaN pada hasil transform — dataset
sumber tidak diubah/diimputasi.

In [12]:
scaler = MinMaxScaler()
scaler.fit(train_df[FEATURE_COLUMNS])

train_scaled = train_df.copy()
validation_scaled = validation_df.copy()
test_scaled = test_df.copy()

train_scaled[FEATURE_COLUMNS] = scaler.transform(train_df[FEATURE_COLUMNS])
validation_scaled[FEATURE_COLUMNS] = scaler.transform(validation_df[FEATURE_COLUMNS])
test_scaled[FEATURE_COLUMNS] = scaler.transform(test_df[FEATURE_COLUMNS])

scaling_detail = {
    "method": "MinMaxScaler",
    "fit_on": "train",
    "feature_columns": FEATURE_COLUMNS,
    "data_min_": {col: float(v) for col, v in zip(FEATURE_COLUMNS, scaler.data_min_)},
    "data_max_": {col: float(v) for col, v in zip(FEATURE_COLUMNS, scaler.data_max_)},
}

scaling_df = pd.DataFrame({
    "feature": FEATURE_COLUMNS,
    "min_train": scaler.data_min_,
    "max_train": scaler.data_max_,
})
scaling_df


,feature,min_train,max_train
0,rr,0.000,236.200
1,tavg,23.700,30.700
2,rh,68.800,97.000
3,cin,-515.072,0.000
4,kindex,-2.700,44.900
5,li,-18.907,24.926
6,tt,24.000,54.700
7,sweat,27.214,374.692


In [13]:
# Verifikasi NaN pada fitur sounding tetap dipertahankan (bukan diimputasi) setelah scaling
for name, raw_subset, scaled_subset in (
    ("train", train_df, train_scaled),
    ("validation", validation_df, validation_scaled),
    ("test", test_df, test_scaled),
):
    n_nan_before = raw_subset[FEATURE_COLUMNS].isna().sum().sum()
    n_nan_after = scaled_subset[FEATURE_COLUMNS].isna().sum().sum()
    assert n_nan_before == n_nan_after, f"Jumlah NaN berubah setelah scaling pada subset {name}!"
    print(f"{name:12s}: NaN sebelum scaling = {n_nan_before}, NaN setelah scaling = {n_nan_after} (konsisten)")


train       : NaN sebelum scaling = 295, NaN setelah scaling = 295 (konsisten)
validation  : NaN sebelum scaling = 45, NaN setelah scaling = 45 (konsisten)
test        : NaN sebelum scaling = 400, NaN setelah scaling = 400 (konsisten)


## Fungsi Pembentukan Sequence

Definisi: data sepanjang *lookback* hari digunakan untuk memprediksi label pada hari berikutnya
(hari ke lookback+1). Jika suatu sequence mengandung minimal satu nilai NaN pada fitur input,
sequence tersebut **dibuang** dari dataset akhir (bukan diimputasi), dan jumlah/persentasenya dicatat.

In [14]:
def build_sequences(df_scaled, lookback, feature_cols, label_col="risk_label_encoded"):
    """Membentuk sequence (X, y) dari dataframe yang sudah di-scale.

    X.shape = (samples, lookback, n_features)
    y.shape = (samples,)

    Sequence yang mengandung NaN pada fitur input dibuang dan dihitung jumlahnya.
    """
    features = df_scaled[feature_cols].to_numpy(dtype=float)
    labels = df_scaled[label_col].to_numpy()
    n = len(df_scaled)
    n_theoretical = max(n - lookback, 0)

    X_list, y_list = [], []
    n_dropped = 0

    for i in range(lookback, n):
        window = features[i - lookback:i]
        target = labels[i]
        if np.isnan(window).any():
            n_dropped += 1
            continue
        X_list.append(window)
        y_list.append(target)

    if X_list:
        X = np.array(X_list, dtype=float)
        y = np.array(y_list)
    else:
        X = np.empty((0, lookback, len(feature_cols)), dtype=float)
        y = np.empty((0,))

    return X, y, n_theoretical, n_dropped


print("Fungsi build_sequences() siap digunakan.")


Fungsi build_sequences() siap digunakan.


## 7. Pembentukan Sequence LB7

Lookback 7 hari digunakan untuk memprediksi label hari ke-8.

In [15]:
LB7 = 7

X_train_lb7, y_train_lb7, n_theo_train_lb7, n_drop_train_lb7 = build_sequences(train_scaled, LB7, FEATURE_COLUMNS)
X_validation_lb7, y_validation_lb7, n_theo_val_lb7, n_drop_val_lb7 = build_sequences(validation_scaled, LB7, FEATURE_COLUMNS)
X_test_lb7, y_test_lb7, n_theo_test_lb7, n_drop_test_lb7 = build_sequences(test_scaled, LB7, FEATURE_COLUMNS)

print("X_train_lb7      :", X_train_lb7.shape, "| y_train_lb7      :", y_train_lb7.shape)
print("X_validation_lb7 :", X_validation_lb7.shape, "| y_validation_lb7 :", y_validation_lb7.shape)
print("X_test_lb7       :", X_test_lb7.shape, "| y_test_lb7       :", y_test_lb7.shape)


X_train_lb7      : (1994, 7, 8) | y_train_lb7      : (1994,)
X_validation_lb7 : (337, 7, 8) | y_validation_lb7 : (337,)
X_test_lb7       : (269, 7, 8) | y_test_lb7       : (269,)


## 8. Pembentukan Sequence LB14

Lookback 14 hari digunakan untuk memprediksi label hari ke-15.

In [16]:
LB14 = 14

X_train_lb14, y_train_lb14, n_theo_train_lb14, n_drop_train_lb14 = build_sequences(train_scaled, LB14, FEATURE_COLUMNS)
X_validation_lb14, y_validation_lb14, n_theo_val_lb14, n_drop_val_lb14 = build_sequences(validation_scaled, LB14, FEATURE_COLUMNS)
X_test_lb14, y_test_lb14, n_theo_test_lb14, n_drop_test_lb14 = build_sequences(test_scaled, LB14, FEATURE_COLUMNS)

print("X_train_lb14      :", X_train_lb14.shape, "| y_train_lb14      :", y_train_lb14.shape)
print("X_validation_lb14 :", X_validation_lb14.shape, "| y_validation_lb14 :", y_validation_lb14.shape)
print("X_test_lb14       :", X_test_lb14.shape, "| y_test_lb14       :", y_test_lb14.shape)


X_train_lb14      : (1858, 14, 8) | y_train_lb14      : (1858,)
X_validation_lb14 : (316, 14, 8) | y_validation_lb14 : (316,)
X_test_lb14       : (255, 14, 8) | y_test_lb14       : (255,)


## 9. Pembentukan Sequence LB30

Lookback 30 hari digunakan untuk memprediksi label hari ke-31.

In [17]:
LB30 = 30

X_train_lb30, y_train_lb30, n_theo_train_lb30, n_drop_train_lb30 = build_sequences(train_scaled, LB30, FEATURE_COLUMNS)
X_validation_lb30, y_validation_lb30, n_theo_val_lb30, n_drop_val_lb30 = build_sequences(validation_scaled, LB30, FEATURE_COLUMNS)
X_test_lb30, y_test_lb30, n_theo_test_lb30, n_drop_test_lb30 = build_sequences(test_scaled, LB30, FEATURE_COLUMNS)

print("X_train_lb30      :", X_train_lb30.shape, "| y_train_lb30      :", y_train_lb30.shape)
print("X_validation_lb30 :", X_validation_lb30.shape, "| y_validation_lb30 :", y_validation_lb30.shape)
print("X_test_lb30       :", X_test_lb30.shape, "| y_test_lb30       :", y_test_lb30.shape)


X_train_lb30      : (1592, 30, 8) | y_train_lb30      : (1592,)
X_validation_lb30 : (268, 30, 8) | y_validation_lb30 : (268,)
X_test_lb30       : (223, 30, 8) | y_test_lb30       : (223,)


## 10. Audit Sequence

Untuk LB7, LB14, dan LB30: jumlah sequence teoritis, valid, dibuang beserta persentasenya, shape
X/y per subset, serta perbandingan distribusi label sebelum dan sesudah pembentukan sequence.

In [18]:
sequence_results = {
    7: {
        "train": (X_train_lb7, y_train_lb7),
        "validation": (X_validation_lb7, y_validation_lb7),
        "test": (X_test_lb7, y_test_lb7),
    },
    14: {
        "train": (X_train_lb14, y_train_lb14),
        "validation": (X_validation_lb14, y_validation_lb14),
        "test": (X_test_lb14, y_test_lb14),
    },
    30: {
        "train": (X_train_lb30, y_train_lb30),
        "validation": (X_validation_lb30, y_validation_lb30),
        "test": (X_test_lb30, y_test_lb30),
    },
}

split_dfs = {"train": train_scaled, "validation": validation_scaled, "test": test_scaled}

audit_records = []
distribution_records = []

for lb, splits in sequence_results.items():
    for split_name, (X, y) in splits.items():
        split_df = split_dfs[split_name]
        n = len(split_df)
        n_theoretical = max(n - lb, 0)
        n_valid = len(y)
        n_dropped = n_theoretical - n_valid
        pct_dropped = (n_dropped / n_theoretical * 100) if n_theoretical > 0 else 0.0

        # Distribusi label sebelum sequence generation (label target teoritis, index lb..n-1)
        target_labels_theoretical = split_df["risk_label"].to_numpy()[lb:]
        dist_before = pd.Series(target_labels_theoretical).value_counts().to_dict()

        # Distribusi label sesudah sequence generation (label pada sequence valid)
        dist_after = pd.Series(y).map(INV_LABEL_MAPPING).value_counts().to_dict() if n_valid > 0 else {}

        audit_records.append({
            "lookback": f"LB{lb}",
            "split": split_name,
            "n_theoretical": n_theoretical,
            "n_valid": n_valid,
            "n_dropped": n_dropped,
            "pct_dropped": round(pct_dropped, 2),
            "X_shape": str(X.shape),
            "y_shape": str(y.shape),
        })
        distribution_records.append({
            "lookback": lb, "split": split_name,
            "dist_before": dist_before, "dist_after": dist_after,
        })

audit_df = pd.DataFrame(audit_records)
audit_df


,lookback,split,n_theoretical,n_valid,n_dropped,pct_dropped,X_shape,y_shape
0,LB7,train,2184,1994,190,8.70,"(1994, 7, 8)","(1994,)"
1,LB7,validation,358,337,21,5.87,"(337, 7, 8)","(337,)"
2,LB7,test,359,269,90,25.07,"(269, 7, 8)","(269,)"
3,LB14,train,2177,1858,319,14.65,"(1858, 14, 8)","(1858,)"
4,LB14,validation,351,316,35,9.97,"(316, 14, 8)","(316,)"
5,LB14,test,352,255,97,27.56,"(255, 14, 8)","(255,)"
6,LB30,train,2161,1592,569,26.33,"(1592, 30, 8)","(1592,)"
7,LB30,validation,335,268,67,20.00,"(268, 30, 8)","(268,)"
8,LB30,test,336,223,113,33.63,"(223, 30, 8)","(223,)"


In [19]:
# Ringkasan total (gabungan train+validation+test) per lookback
audit_totals = {}
for lb in (7, 14, 30):
    rows = audit_df[audit_df["lookback"] == f"LB{lb}"]
    n_theo = int(rows["n_theoretical"].sum())
    n_valid = int(rows["n_valid"].sum())
    n_dropped = int(rows["n_dropped"].sum())
    pct_dropped = (n_dropped / n_theo * 100) if n_theo > 0 else 0.0
    audit_totals[lb] = {
        "n_theoretical": n_theo, "n_valid": n_valid,
        "n_dropped": n_dropped, "pct_dropped": round(pct_dropped, 2),
    }
    print(f"LB{lb:<3d}: teoritis={n_theo:5d} | valid={n_valid:5d} | dibuang={n_dropped:4d} ({pct_dropped:.2f}%)")

audit_totals


LB7  : teoritis= 2901 | valid= 2600 | dibuang= 301 (10.38%)
LB14 : teoritis= 2880 | valid= 2429 | dibuang= 451 (15.66%)
LB30 : teoritis= 2832 | valid= 2083 | dibuang= 749 (26.45%)


{7: {'n_theoretical': 2901,
  'n_valid': 2600,
  'n_dropped': 301,
  'pct_dropped': 10.38},
 14: {'n_theoretical': 2880,
  'n_valid': 2429,
  'n_dropped': 451,
  'pct_dropped': 15.66},
 30: {'n_theoretical': 2832,
  'n_valid': 2083,
  'n_dropped': 749,
  'pct_dropped': 26.45}}

In [20]:
# Perbandingan distribusi label sebelum vs sesudah sequence generation, per lookback (gabungan subset)
def analyze_distribution_shift(dist_before, dist_after, threshold_pp=2.0):
    total_before = sum(dist_before.values())
    total_after = sum(dist_after.values())
    lines = []
    max_diff = 0.0
    for label in LABEL_MAPPING:
        before_pct = (dist_before.get(label, 0) / total_before * 100) if total_before > 0 else 0.0
        after_pct = (dist_after.get(label, 0) / total_after * 100) if total_after > 0 else 0.0
        diff = after_pct - before_pct
        max_diff = max(max_diff, abs(diff))
        lines.append(f"{label}: {before_pct:.2f}% -> {after_pct:.2f}% (selisih {diff:+.2f} pp)")
    verdict = "tidak signifikan" if max_diff < threshold_pp else "signifikan"
    return lines, verdict, max_diff


lookback_distribution_summary = {}
for lb in (7, 14, 30):
    agg_before, agg_after = {}, {}
    for rec in distribution_records:
        if rec["lookback"] == lb:
            for k, v in rec["dist_before"].items():
                agg_before[k] = agg_before.get(k, 0) + v
            for k, v in rec["dist_after"].items():
                agg_after[k] = agg_after.get(k, 0) + v
    lines, verdict, max_diff = analyze_distribution_shift(agg_before, agg_after)
    lookback_distribution_summary[lb] = {
        "before": agg_before, "after": agg_after,
        "lines": lines, "verdict": verdict, "max_diff": round(max_diff, 2),
    }
    print(f"\nLB{lb} — pergeseran distribusi label maksimum: {max_diff:.2f} pp ({verdict})")
    for line in lines:
        print("  -", line)



LB7 — pergeseran distribusi label maksimum: 0.57 pp (tidak signifikan)
  - Rendah: 80.01% -> 80.58% (selisih +0.57 pp)
  - Sedang: 12.69% -> 12.15% (selisih -0.53 pp)
  - Tinggi: 5.76% -> 5.85% (selisih +0.09 pp)
  - Sangat Tinggi: 1.55% -> 1.42% (selisih -0.13 pp)

LB14 — pergeseran distribusi label maksimum: 0.41 pp (tidak signifikan)
  - Rendah: 80.00% -> 80.28% (selisih +0.28 pp)
  - Sedang: 12.67% -> 12.27% (selisih -0.41 pp)
  - Tinggi: 5.76% -> 5.97% (selisih +0.21 pp)
  - Sangat Tinggi: 1.56% -> 1.48% (selisih -0.08 pp)

LB30 — pergeseran distribusi label maksimum: 0.62 pp (tidak signifikan)
  - Rendah: 80.23% -> 80.70% (selisih +0.47 pp)
  - Sedang: 12.43% -> 11.81% (selisih -0.62 pp)
  - Tinggi: 5.76% -> 6.00% (selisih +0.25 pp)
  - Sangat Tinggi: 1.59% -> 1.49% (selisih -0.10 pp)


## 11. Penyimpanan Artefak

Menyimpan seluruh artefak Stage 10 ke dalam folder `stage10/`: `scaler.pkl`, array `.npy` untuk
LB7/LB14/LB30, serta file `.csv` hasil split (mentah dan ter-scaling).

In [21]:
BASE_DIR = "stage10"
os.makedirs(BASE_DIR, exist_ok=True)
for lb in (7, 14, 30):
    os.makedirs(os.path.join(BASE_DIR, f"LB{lb}"), exist_ok=True)

# Simpan scaler
joblib.dump(scaler, os.path.join(BASE_DIR, "scaler.pkl"))

# Simpan array sequence per lookback
for lb, splits in sequence_results.items():
    lb_dir = os.path.join(BASE_DIR, f"LB{lb}")
    for split_name, (X, y) in splits.items():
        np.save(os.path.join(lb_dir, f"X_{split_name}.npy"), X)
        np.save(os.path.join(lb_dir, f"y_{split_name}.npy"), y)

# Simpan split mentah (belum di-scale)
train_df.to_csv(os.path.join(BASE_DIR, "train.csv"), index=False)
validation_df.to_csv(os.path.join(BASE_DIR, "validation.csv"), index=False)
test_df.to_csv(os.path.join(BASE_DIR, "test.csv"), index=False)

# Simpan split hasil scaling
train_scaled.to_csv(os.path.join(BASE_DIR, "train_scaled.csv"), index=False)
validation_scaled.to_csv(os.path.join(BASE_DIR, "validation_scaled.csv"), index=False)
test_scaled.to_csv(os.path.join(BASE_DIR, "test_scaled.csv"), index=False)

print("Seluruh artefak berhasil disimpan pada folder:", os.path.abspath(BASE_DIR))
for root, dirs, files in os.walk(BASE_DIR):
    for f in sorted(files):
        print(" -", os.path.relpath(os.path.join(root, f), BASE_DIR))


Seluruh artefak berhasil disimpan pada folder: /home/claude/work/stage10
 - scaler.pkl
 - test.csv
 - test_scaled.csv
 - train.csv
 - train_scaled.csv
 - validation.csv
 - validation_scaled.csv
 - LB30/X_test.npy
 - LB30/X_train.npy
 - LB30/X_validation.npy
 - LB30/y_test.npy
 - LB30/y_train.npy
 - LB30/y_validation.npy
 - LB7/X_test.npy
 - LB7/X_train.npy
 - LB7/X_validation.npy
 - LB7/y_test.npy
 - LB7/y_train.npy
 - LB7/y_validation.npy
 - LB14/X_test.npy
 - LB14/X_train.npy
 - LB14/X_validation.npy
 - LB14/y_test.npy
 - LB14/y_train.npy
 - LB14/y_validation.npy


## 12. Pembuatan STAGE10_REPORT.md

Menghasilkan laporan otomatis berdasarkan seluruh hasil validasi dan audit di atas.

In [22]:
def fmt_dict_table(d, key_header, val_header, val_fmt="{}"):
    lines = [f"| {key_header} | {val_header} |", "|---|---|"]
    for k, v in d.items():
        lines.append(f"| {k} | {val_fmt.format(v)} |")
    return "\n".join(lines)


report_lines = []
report_lines.append("# STAGE 10 REPORT - Dataset Preparation for LSTM Flood Risk Classification\n")
report_lines.append(f"**Input:** `{INPUT_FILE}` ({validation_results['n_rows']} baris, {validation_results['n_cols']} kolom)  ")
report_lines.append("**Sifat tahap:** Persiapan dataset (split, encoding, scaling, sequence generation) — "
                     "dataset sumber tidak diubah/diimputasi.  ")
report_lines.append("**Referensi:** `STAGE9_REPORT.md`\n")
report_lines.append("---\n")

# 1. Ringkasan dataset input
report_lines.append("## 1. Ringkasan Dataset Input\n")
report_lines.append(f"- Jumlah baris: **{validation_results['n_rows']}**")
report_lines.append(f"- Jumlah kolom: **{validation_results['n_cols']}**")
report_lines.append(f"- Rentang tanggal: **{validation_results['date_min']} s.d. {validation_results['date_max']}**")
report_lines.append(f"- Feature model (8): `{', '.join(FEATURE_COLUMNS)}`")
report_lines.append(f"- Target: `{TARGET_COLUMN}`")
report_lines.append("- Kolom `cape`, `nominal_date`, `observation_datetime`, `label_source`, "
                     "`missing_group` **tidak digunakan** sebagai feature model.\n")

# 2. Hasil validasi dataset
report_lines.append("## 2. Hasil Validasi Dataset\n")
report_lines.append("**Tipe data per kolom:**\n")
report_lines.append(fmt_dict_table(validation_results["dtypes"], "Kolom", "Tipe Data"))
report_lines.append("")
report_lines.append(f"- Gap tanggal kalender: **{validation_results['date_gap']}** "
                     f"(dari {validation_results['expected_days']} hari yang diharapkan)")
report_lines.append(f"- Duplicate row: **{validation_results['duplicate_rows']}**")
report_lines.append(f"- Duplicate date: **{validation_results['duplicate_dates']}**\n")
report_lines.append("**Missing value per kolom:**\n")
report_lines.append("| Kolom | Missing | Persen |")
report_lines.append("|---|---|---|")
for col, info in validation_results["missing_per_column"].items():
    report_lines.append(f"| {col} | {info['missing']} | {info['persen']}% |")
report_lines.append("")

# 3. Ringkasan split
report_lines.append("## 3. Ringkasan Split Kronologis\n")
report_lines.append("| Subset | Jumlah Baris | Rentang Tanggal |")
report_lines.append("|---|---|---|")
for split_name, info in split_summary.items():
    report_lines.append(f"| {split_name.capitalize()} | {info['n_rows']} | {info['date_min']} s.d. {info['date_max']} |")
report_lines.append("\nMetode split: **kronologis**, berdasarkan rentang tanggal tetap. Tidak dilakukan "
                     "random split, shuffle, maupun stratified split.\n")

# 4. Distribusi label
report_lines.append("## 4. Distribusi Label\n")
report_lines.append("**Distribusi label keseluruhan dataset:**\n")
report_lines.append("| risk_label | Jumlah | Persen |")
report_lines.append("|---|---|---|")
for label, info in validation_results["label_distribution"].items():
    report_lines.append(f"| {label} | {info['jumlah']} | {info['persen']}% |")
report_lines.append("")
report_lines.append(f"**Label encoding:** `{json.dumps(LABEL_MAPPING, ensure_ascii=False)}`\n")

report_lines.append("**Distribusi label per subset:**\n")
report_lines.append("| Subset | Rendah | Sedang | Tinggi | Sangat Tinggi |")
report_lines.append("|---|---|---|---|---|")
for split_name, split_df_ in (("Train", train_df), ("Validation", validation_df), ("Test", test_df)):
    counts = split_df_["risk_label"].value_counts()
    row = [str(counts.get(lbl, 0)) for lbl in ["Rendah", "Sedang", "Tinggi", "Sangat Tinggi"]]
    report_lines.append(f"| {split_name} | " + " | ".join(row) + " |")
report_lines.append("")

# 5. Detail scaling
report_lines.append("## 5. Detail Scaling\n")
report_lines.append(f"- Metode: **{scaling_detail['method']}**")
report_lines.append("- Scaler di-*fit* **hanya pada data Train**, kemudian digunakan untuk "
                     "*transform* Train, Validation, dan Test.")
report_lines.append("- Tidak dilakukan fit pada Validation maupun Test.")
report_lines.append("- Nilai NaN pada fitur sounding diabaikan saat fit dan tetap dipertahankan "
                     "sebagai NaN setelah transform (tidak diimputasi).\n")
report_lines.append("**Parameter scaler (min/max dari data Train):**\n")
report_lines.append("| Feature | Min (Train) | Max (Train) |")
report_lines.append("|---|---|---|")
for col in FEATURE_COLUMNS:
    report_lines.append(f"| {col} | {scaling_detail['data_min_'][col]:.4f} | {scaling_detail['data_max_'][col]:.4f} |")
report_lines.append("\nArtefak scaler disimpan sebagai `scaler.pkl`.\n")

# 6-8. Audit LB7/LB14/LB30
for lb in (7, 14, 30):
    report_lines.append(f"## {5 + [7,14,30].index(lb) + 1}. Audit LB{lb}\n")
    rows = audit_df[audit_df["lookback"] == f"LB{lb}"]
    report_lines.append("| Subset | Sequence Teoritis | Sequence Valid | Sequence Dibuang | % Dibuang | X shape | y shape |")
    report_lines.append("|---|---|---|---|---|---|---|")
    for _, r in rows.iterrows():
        report_lines.append(
            f"| {r['split'].capitalize()} | {r['n_theoretical']} | {r['n_valid']} | "
            f"{r['n_dropped']} | {r['pct_dropped']}% | {r['X_shape']} | {r['y_shape']} |"
        )
    tot = audit_totals[lb]
    report_lines.append(
        f"| **Total** | **{tot['n_theoretical']}** | **{tot['n_valid']}** | "
        f"**{tot['n_dropped']}** | **{tot['pct_dropped']}%** | - | - |"
    )
    report_lines.append("")
    dist = lookback_distribution_summary[lb]
    report_lines.append(f"**Distribusi label sebelum vs sesudah sequence generation (LB{lb}, gabungan seluruh subset):**\n")
    for line in dist["lines"]:
        report_lines.append(f"- {line}")
    report_lines.append(f"\nPergeseran distribusi label maksimum: **{dist['max_diff']} poin persentase** "
                         f"→ dinilai **{dist['verdict']}** (ambang batas 2.0 pp).\n")

# 9. Sequence yang dibuang
report_lines.append("## 9. Sequence yang Dibuang\n")
report_lines.append("| Lookback | Sequence Dibuang | % dari Sequence Teoritis |")
report_lines.append("|---|---|---|")
for lb in (7, 14, 30):
    tot = audit_totals[lb]
    report_lines.append(f"| LB{lb} | {tot['n_dropped']} | {tot['pct_dropped']}% |")
report_lines.append("\nSequence dibuang murni disebabkan oleh keberadaan minimal satu nilai NaN pada "
                     "fitur sounding (`cin`, `kindex`, `li`, `tt`, `sweat`) di dalam window lookback-nya. "
                     "Tidak ada baris dataset sumber yang dihapus atau diimputasi.\n")

# 10. Analisis dampak missing value
report_lines.append("## 10. Analisis Dampak Missing Value\n")
n_missing_rows = validation_results["missing_per_column"]["cin"]["missing"]
pct_missing_rows = validation_results["missing_per_column"]["cin"]["persen"]
report_lines.append(f"- Dataset sumber memiliki **{n_missing_rows} baris ({pct_missing_rows}%)** dengan "
                     "missing value pada kolom indeks sounding (`cin`, `kindex`, `li`, `tt`, `sweat`), "
                     "konsisten dengan temuan Stage 9.")
report_lines.append("- Semakin besar lookback window, semakin besar pula peluang suatu sequence "
                     "'menyentuh' hari dengan missing value, sehingga jumlah dan persentase sequence "
                     "yang dibuang meningkat seiring bertambahnya lookback:")
for lb in (7, 14, 30):
    tot = audit_totals[lb]
    report_lines.append(f"  - LB{lb}: {tot['n_dropped']} sequence dibuang ({tot['pct_dropped']}%)")
report_lines.append("- Kebijakan *drop* (bukan imputasi) dipilih untuk menjaga integritas dataset sumber "
                     "sesuai batasan metodologi Stage 10.\n")

# 11. Daftar artefak
report_lines.append("## 11. Daftar Artefak yang Dihasilkan\n")
report_lines.append("```")
report_lines.append("stage10/")
report_lines.append("├── scaler.pkl")
for lb in (7, 14, 30):
    report_lines.append(f"├── LB{lb}/")
    report_lines.append("│   ├── X_train.npy")
    report_lines.append("│   ├── y_train.npy")
    report_lines.append("│   ├── X_validation.npy")
    report_lines.append("│   ├── y_validation.npy")
    report_lines.append("│   ├── X_test.npy")
    report_lines.append("│   └── y_test.npy")
report_lines.append("├── train.csv")
report_lines.append("├── validation.csv")
report_lines.append("├── test.csv")
report_lines.append("├── train_scaled.csv")
report_lines.append("├── validation_scaled.csv")
report_lines.append("├── test_scaled.csv")
report_lines.append("└── STAGE10_REPORT.md")
report_lines.append("```\n")

# 12. Kesimpulan kesiapan Stage 11
report_lines.append("## 12. Kesimpulan Kesiapan Stage 11\n")
report_lines.append("| Aspek | Status | Catatan |")
report_lines.append("|---|---|---|")
report_lines.append("| Split kronologis Train/Validation/Test | Siap | Tidak ada tumpang-tindih baris antar subset |")
report_lines.append("| Label encoding | Siap | Seluruh label ter-encode tanpa nilai kosong |")
report_lines.append(f"| Feature scaling (MinMaxScaler) | Siap | Fit hanya pada Train; NaN dipertahankan (tidak diimputasi) |")
report_lines.append(f"| Sequence LB7 | Siap | {audit_totals[7]['n_valid']} sequence valid, {audit_totals[7]['pct_dropped']}% dibuang |")
report_lines.append(f"| Sequence LB14 | Siap | {audit_totals[14]['n_valid']} sequence valid, {audit_totals[14]['pct_dropped']}% dibuang |")
report_lines.append(f"| Sequence LB30 | Siap | {audit_totals[30]['n_valid']} sequence valid, {audit_totals[30]['pct_dropped']}% dibuang |")
report_lines.append("| Class imbalance (dibawa dari Stage 9) | Belum ditangani | Ditangani pada tahap training "
                     "(mis. class_weight), bukan pada Stage 10 sesuai batasan tahap ini |")
report_lines.append("\nSeluruh artefak Stage 10 (scaler, sequence LB7/LB14/LB30, serta file CSV split "
                     "mentah dan ter-scaling) telah berhasil dibuat dan disimpan. Dataset **siap "
                     "dilanjutkan ke Stage 11** dengan catatan bahwa penanganan class imbalance (yang "
                     "teridentifikasi pada Stage 9 dengan rasio 51,8:1) merupakan tanggung jawab tahap "
                     "training model, bukan bagian dari Stage 10.\n")
report_lines.append("---\n")
report_lines.append("## Validasi Kepatuhan Batasan Stage 10\n")
report_lines.append("| Batasan | Status |")
report_lines.append("|---|---|")
report_lines.append("| Split kronologis (bukan random/shuffle/stratified) | Dipatuhi |")
report_lines.append("| Scaler fit hanya pada Train | Dipatuhi |")
report_lines.append("| Tidak ada fillna/interpolasi/imputasi pada dataset sumber | Dipatuhi |")
report_lines.append("| Sequence mengandung NaN dibuang, dicatat jumlah & persentase | Dipatuhi |")
report_lines.append("| Tidak ada training model / hyperparameter tuning | Dipatuhi |")
report_lines.append("| Tidak ada class balancing / SMOTE / oversampling / undersampling | Dipatuhi |")
report_lines.append("| Tidak ada feature selection | Dipatuhi |")
report_lines.append("| Tidak ada evaluasi model (confusion matrix, precision, recall, F1, ROC/AUC) | Dipatuhi |")
report_lines.append("\n## **STAGE 10 = PASS**\n")

report_text = "\n".join(report_lines)

report_path = os.path.join(BASE_DIR, "STAGE10_REPORT.md")
with open(report_path, "w", encoding="utf-8") as f:
    f.write(report_text)

print(f"STAGE10_REPORT.md berhasil dibuat di: {os.path.abspath(report_path)}")
print(f"Panjang laporan: {len(report_text)} karakter")


STAGE10_REPORT.md berhasil dibuat di: /home/claude/work/stage10/STAGE10_REPORT.md
Panjang laporan: 8958 karakter


In [23]:
# Tampilkan pratinjau laporan
print(report_text[:3000])
print("\n... (lihat file lengkap di stage10/STAGE10_REPORT.md) ...")


# STAGE 10 REPORT - Dataset Preparation for LSTM Flood Risk Classification

**Input:** `08_labeled_dataset.csv` (2922 baris, 15 kolom)  
**Sifat tahap:** Persiapan dataset (split, encoding, scaling, sequence generation) — dataset sumber tidak diubah/diimputasi.  
**Referensi:** `STAGE9_REPORT.md`

---

## 1. Ringkasan Dataset Input

- Jumlah baris: **2922**
- Jumlah kolom: **15**
- Rentang tanggal: **2017-01-01 s.d. 2024-12-31**
- Feature model (8): `rr, tavg, rh, cin, kindex, li, tt, sweat`
- Target: `risk_label`
- Kolom `cape`, `nominal_date`, `observation_datetime`, `label_source`, `missing_group` **tidak digunakan** sebagai feature model.

## 2. Hasil Validasi Dataset

**Tipe data per kolom:**

| Kolom | Tipe Data |
|---|---|
| date | datetime64[us] |
| selected_hour | str |
| selection_status | str |
| rr | float64 |
| tavg | float64 |
| rh | float64 |
| cin | float64 |
| kindex | float64 |
| li | float64 |
| tt | float64 |
| sweat | float64 |
| cape | float64 |
| missing_group | 